# PyTorch: ViT using pre-trained weights

In [ ]:
import torch
import random
from torchinfo import summary
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchmetrics.classification import MulticlassAccuracy
from common import CV_DATASETS_DIR
import common.torch as ct

In [ ]:
ct.set_default_seed()
ct.set_default_optimizations()
device = ct.get_optimal_device()

In [ ]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

In [ ]:
# Hyperparameters
BATCH_SIZE = 32
N_EPOCHS = 5
# Other parameters
IMAGE_SIZE = (224,224)

## Prepare Datasets

In [ ]:
DATASET_PATH = CV_DATASETS_DIR/"animals"

tr_dataset = datasets.OxfordIIITPet(root=DATASET_PATH,
                                    transform=ViT_B_16_Weights.DEFAULT.transforms(),
                                    split="trainval",
                                    download=True)
ts_dataset = datasets.OxfordIIITPet(root=DATASET_PATH,
                                    transform=ViT_B_16_Weights.DEFAULT.transforms(),
                                    split="test",
                                    download=True)

assert tr_dataset.class_to_idx == ts_dataset.class_to_idx, "Train and Test class indices mismatch!"
len(tr_dataset), len(ts_dataset)

In [ ]:
tr_dl = DataLoader(tr_dataset, batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
ts_dl = DataLoader(ts_dataset, batch_size=BATCH_SIZE, num_workers=2)
len(tr_dl), len(ts_dl)

In [ ]:
n_classes = len(tr_dataset.classes)
n_classes

## Define Model

In [ ]:
# Create a model using pretrained weights
model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT).to(device)

for parameter in model.parameters():
    parameter.requires_grad = False

model.heads = nn.Sequential(
    nn.Linear(model.heads[0].in_features, n_classes)
).to(device)

In [ ]:
summary(model=model,
        input_size=(1, 3)+IMAGE_SIZE,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

## Train Model

In [ ]:
logs_dir, writer = ct.get_summary_writer("pt_vit_transfer_learning", "vit_with_weights", "5_epochs")
logs_dir

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), weight_decay=0.1)
accuracy = MulticlassAccuracy(num_classes=n_classes).to(device)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

In [ ]:
ct.train(model=model,
         tr_dl=tr_dl,
         ts_dl=ts_dl,
         optimizer=optimizer,
         criterion=criterion,
         metric=accuracy,
         n_epochs=N_EPOCHS,
         writer=writer,
         device=device,
         scheduler=scheduler)